# Business-oriented Smile Simulation (Preserved Lip Area)

This Notebook will demonstrate how to generate step-by-step smile images based on the initial smile photo and the orthodontic stepwise plan.

In [ ]:
import os
import glob
import base64
import time
import requests
import json
import trimesh
import urllib
import numpy as np
import DracoPy
from PIL import Image
from io import BytesIO
import cv2
import glob
import matplotlib.pyplot as plt

## Define call rules
Please modify the following code blocks based on the information you obtained from us.

In [ ]:
# Chohotech service request URL, sent with the API documentation.
base_url = "<service request URL>"

# Chohotech file service URL, sent with the API documentation.
file_server_url = "<service file server URL>"

# The authentication header must be passed in. Please keep the TOKEN confidential!!! If it is leaked, please contact us immediately to reset it. All tasks using this TOKEN will be charged to your account.
zh_token = "<your company's service Token, sent with the contact>" # All API calls must be authenticated with the token.

user_group = "APIClient" # User group, usually named APIClient.

# Your company's user_id, sent with the API documentation.
user_id = "<your company's user_id>"

# If you have received creds.json, it will be read directly below.
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

In [ ]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # Must specify postfix, i.e., the file extension
                        headers={"X-ZH-TOKEN": zh_token}) # Get the signed upload URL
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # Returns a single string JSON "string", can also use json.loads(resp.text)

    resp = requests.put(upload_url, data) # No auth header is needed for uploading to the cloud storage service

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def retrieve_mesh(mesh_file_json):
    resp = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": mesh_file_json['data']}),
                        headers={"X-ZH-TOKEN": zh_token})
    return trimesh.load(trimesh.util.wrap_as_stream(resp.content), file_type=mesh_file_json['type'])

def show_img(urn):
    imbytes = requests.get(file_server_url + f"/file/download?" + urllib.parse.urlencode({
                        "urn": urn}),
                        headers={"X-ZH-TOKEN": zh_token}).content
    imarray = np.asarray(bytearray(imbytes), dtype=np.uint8)
    plt.imshow(cv2.imdecode(imarray, cv2.IMREAD_COLOR)[...,::-1])

##  Automatic Form Generation and Form-Based Teeth Arrangement
please refer to: https://www.chohotech.com/docs/cloud-en/#/workflow/oral-arrangement-medical-1

In [ ]:
json_call = {
  "spec_group": "mesh-processing",
  "spec_name": "oral-arrangement-medical",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "upper_mesh": {"type":"drc", "data": upload_file("upper_jaw_scan.drc")},
      "lower_mesh": {"type":"ply", "data": upload_file("lower_jaw_scan.ply")},
      "ceph": upload_file("ceph.jpg"),
      "smile_photo": upload_file("face_smile.jpg"),
  },
  'output_config': {
    "u_teeth_comp": {"type": "ply"},
    "l_teeth_comp": {"type": "ply"}
  }
}
result_arrangement = run_job_and_get_results(json_call, 1200)

## Auto-Step

In [ ]:
json_call = {
  "spec_group": "mesh-processing", # he invoked workflow group is sent along with the API documentation.
  "spec_name": "auto-step", # he invoked workflow group is sent along with the API documentation.
  "spec_version": "1.0-snapshot", # he invoked workflow group is sent along with the API documentation.
  "user_group": user_group,
  "user_id": user_id
}
json_call["input_data"] = {
    "upper_teeth_dict": result_arrangement["u_teeth_comp"],
    "upper_align_matrix": result_arrangement["u_align_matrix"],
    "upper_axis_matrix_dict": result_arrangement["u_axis"],
    "lower_teeth_dict": result_arrangement["l_teeth_comp"],
    "lower_align_matrix": result_arrangement["l_align_matrix"],
    "lower_axis_matrix_dict": result_arrangement["l_axis"],
    "transformation_dict": result_arrangement['transformation_dict']
}
result_step = run_job_and_get_results(json_call, 2200)

In [ ]:
original_comp = {**{k: retrieve_mesh(v) for k, v in result_arrangement["u_teeth_comp"].items()},
                 **{k: retrieve_mesh(v) for k, v in result_arrangement["l_teeth_comp"].items()}}
def show_step(step_index):
    result_mesh = None
    for k, m in original_comp.items():
        if k in result_step['result']['step_dict'][step_index]:
            result_mesh += m.copy().apply_transform(result_step['result']['step_dict'][step_index][k])
    return result_mesh

# Generate smile simulations for each step

### Assemble the step dictionary

Note: You must include 'step_0' representing the initial position for registration with the smile image

In [ ]:
step_0 = {}
step_dict = {}

for k,v in result_arrangement["transformation_dict"].items():
    step_0[k] = np.eye(4,4).tolist()
step_dict['step_0'] = step_0

for i, step_transformation in enumerate(result_step["result"]['step_dict']):
    step_dict[f'step_{i+1}'] = step_transformation

### Smile-Sim-to-Business
This interface simulates a patient’s smile after orthodontic treatment using the patient’s current smile photo and the 3D dental models manually adjusted by the dentist.
Please refer to: https://www.chohotech.com/docs/cloud-en/#/module/smile-sim-to-business-1

In [ ]:
json_call = {
  "spec_group": "smile",
  "spec_name": "smile-sim-to-business",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {
      "image": upload_file("face_smile.jpg"),
      "meshes": {**result_arrangement["u_teeth_comp"], **result_arrangement["l_teeth_comp"]},
      "step": step_dict
  }
}
result_sim = run_job_and_get_results(json_call, 1000)

## Visualization

In [ ]:
for idx, urn in result_sim['result']['image'].items():
    plt.figure()
    plt.title(idx)
    show_img(result_sim['result']['image'][idx])

In [ ]:
show_step(10)